# Base Place Recognition Pipeline 

Test Place Recognition on the 3DSSG dataset using `opr.pipelines`

In [18]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex, SequencePlaceRecognitionPipeline

from gsloc.inference.pr_infer import PRInferencer
from gsloc.models import graph_encoder as network
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

## Create dataset object

In [7]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
index_path = "/home/kartashov_ga/projects/tests/gsloc/26-04-17/graph"

In [8]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [9]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index_path,
    rebuild_meta=False,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False,
    scene_filter_mode="listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
)
# You can create your own dataloader for index generation
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

2026-04-16 10:40:42.668 | INFO     | gsloc.datasets.three_rscan:__init__:275 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-04-16 10:40:42.669 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:193 - Scanning 3rscan dataset for 30 selected scenes...
2026-04-16 10:41:01.831 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:226 - Scanned 9449 rows


## Create model

In [10]:
# model = MegaLoc()
# model.eval()
# ``MultiModalVPRGraphEncoder`` layout matches ``best_model.pth``; MegaLoc weights are not in the ckpt.
weights_path = Path("/home/kartashov_ga/projects/GSLoc/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

graph_enc = network.VPRGraphEncoder(
    in_dim=4,
    hidden_dim=256,
    n_layers=1,
    num_node_classes=528 + 1,
    num_edge_classes=41,
    node_emb_dim=64,
    edge_emb_dim=64,
    proj_dim=256,
)
model = network.MultiModalVPRGraphEncoder(
    graph_encoder=graph_enc,
    image_encoder=MegaLoc(),
    image_out_dim=8448,
    fusion_dim=8448,
    mode="graph",
)
missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
if unexpected:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected}")
# ``missing`` lists ``image_encoder.*`` (hub MegaLoc), which the checkpoint does not store.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MultiModalVPRGraphEncoder(
  (graph_encoder): VPRGraphEncoder(
    (node_emb): Embedding(529, 64)
    (edge_emb): Embedding(41, 64)
    (edge_proj): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
    (input_mlp): Sequential(
      (0): Linear(in_features=68, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): ReLU(inplace=True)
    )
    (convs): ModuleList(
      (0): GINEConv(nn=Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=256, out_features=256, bias=True)
      ))
    )
    (act): ReLU(inplace=True)
    (drop): Dropout(p=0, inplace=False)
    (proj): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_featur

## Create Index (files that are used to do retrievel based on database)

In [11]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model,
    rebuild_meta=False,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-04-16 10:41:11.846 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 9,449 rows to /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph/meta.parquet
2026-04-16 10:41:11.847 | INFO     | mmpr.inference.index:generate:389 - meta.parquet file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph
  0%|          | 0/394 [00:00<?, ?it/s]

100%|██████████| 394/394 [01:04<00:00,  6.06it/s]
2026-04-16 10:42:16.823 | INFO     | mmpr.inference.index:generate:416 - descriptors.npy file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph
2026-04-16 10:42:16.826 | INFO     | mmpr.inference.index:generate:436 - schema.json file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph


Index created at /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph
Index size: 9449, dim: 256 metric: l2


# Test PlaceRecognitionPipeline

In [12]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    # k=50,
)


seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    max_window=25,
    per_frame_k=20,
    final_k=50,
    descriptor_agg="mean",
)

In [13]:
query_cache_path = Path(index_path) / "query_cache"

three_rscan_q = ThreeRScan(
    dataset_root="/mnt/external_usb_hdd/6YL/Datasets/3RScan",
    meta_path = query_cache_path,
    # save_meta=True,
    rebuild_meta=False,
    # limit=10000,
    image_transform=image_transform_fn,
    scene_filter_mode="same_room_excluding_listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
)

2026-04-16 10:42:23.147 | INFO     | gsloc.datasets.three_rscan:__init__:275 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-04-16 10:42:23.147 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:193 - Scanning 3rscan dataset for 93 selected scenes...
2026-04-16 10:42:45.621 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:226 - Scanned 21013 rows


In [14]:
inferencer = PRInferencer(
    pr_pipeline=pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=25,
    device="cuda"
)

In [15]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "test.npz", frames=frames)
# frames = inferencer.load(query_cache_path / "test.npz")

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [02:36<00:00,  8.39it/s]
2026-04-16 10:45:34.320 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/tests/gsloc/26-04-17/graph/query_cache/meta.parquet


In [11]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    include_per_query=False
)

  0%|          | 0/21013 [00:00<?, ?it/s]

first time building scene to room map


/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/three_rscan.py:407: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 21013/21013 [00:20<00:00, 1000.71it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,7379,0.351164,35.116357
1,5,21013,9694,0.461333,46.133346
2,10,21013,10903,0.518869,51.886927
3,25,21013,12880,0.612954,61.295389


In [11]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 6837.27it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18276,0.869747,86.974730
1,5,21013,19480,0.927045,92.704516
2,10,21013,19916,0.947794,94.779422
3,25,21013,20372,0.969495,96.949507


In [16]:
result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=range(1, 25, 2),
    per_frame_k_used=10,
    save_dir=query_cache_path / "seq_pr_benchmark"
)

  0%|          | 0/12 [00:00<?, ?it/s]

starting sequence fusion
rankings creation started
recall@k calculation started
fused rankings preparation started


  8%|▊         | 1/12 [00:37<06:55, 37.80s/it]

starting sequence fusion
rankings creation started
recall@k calculation started
fused rankings preparation started


 17%|█▋        | 2/12 [01:52<09:55, 59.59s/it]

...

fused rankings preparation started
starting sequence fusion
rankings creation started
recall@k calculation started


 92%|█████████▏| 11/12 [15:15<01:29, 89.26s/it]

fused rankings preparation started
starting sequence fusion
rankings creation started
recall@k calculation started


100%|██████████| 12/12 [16:45<00:00, 83.80s/it]

fused rankings preparation started


In [ ]:
result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=range(1, 25, 2),
    per_frame_k_used=10,
    save_dir=query_cache_path / "seq_pr_benchmark"
)

  0%|          | 0/12 [00:00<?, ?it/s]

Building valid subset...


  8%|▊         | 1/12 [00:37<06:52, 37.46s/it]

Building valid subset...


 17%|█▋        | 2/12 [01:45<09:16, 55.68s/it]

...

Building valid subset...


 92%|█████████▏| 11/12 [23:47<02:39, 159.84s/it]

Building valid subset...


100%|██████████| 12/12 [26:39<00:00, 133.30s/it]


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total
0,1,0.946573,0.899766,0.869747,0.927045,0.947794,0.947794,21013,21013
1,3,0.929119,0.871144,0.890925,0.939990,0.958597,0.977205,21013,21013
2,5,0.920036,0.859249,0.900538,0.945415,0.963070,0.983201,21013,21013
3,7,0.914474,0.853311,0.905630,0.949317,0.966687,0.986199,21013,21013
4,9,0.912266,0.853387,0.909056,0.952220,0.969019,0.987912,21013,21013
5,11,0.912136,0.857502,0.910532,0.953838,0.970875,0.989721,21013,21013
6,13,0.913075,0.863204,0.912245,0.954790,0.971779,0.990197,21013,21013
7,15,0.914181,0.869355,0.913149,0.954933,0.972017,0.989863,21013,21013
8,17,0.915109,0.875215,0.913006,0.954790,0.972303,0.989292,21013,21013
9,19,0.915807,0.880351,0.912388,0.954171,0.972541,0.988483,21013,21013


#Graph random report

In [23]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w' and metrics).
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = "all"
    # precompute simple mean once
    # group = summary_all.groupby("w", as_index=False)
    # mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()

    for m in metrics:
        if m not in df.columns:
            continue
        fig = px.line(df, x="w", y=m, title=f"{map_name}: {m} vs sequence length (w)", markers=True)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay simple mean across maps
        # if m in mean_by_w.columns:
        #     fig.add_trace(
        #         go.Scatter(
        #             x=mean_by_w["w"],
        #             y=mean_by_w[m].astype(float),
        #             mode="lines",
        #             name="mean",
        #             line=dict(color="green", dash="dash"),
        #             showlegend=True,
        #         )
        #     )

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )
            fig.add_trace(
                go.Scatter(
                    x=wmean_series["w"],
                    y=wmean_series[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [24]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQMFBwkLDQ8RExUX', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('Zr4+HhwNxz+1HEXGPfLFPzNrZ83U/8' ... 'i0MY/rxD/7W46qBrjEPzl3ID6niMQ/'),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.18008758045970125]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [16]:
seq_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose", 
        "trans_tol_m": 4, 
        "rot_tol_deg": 90
        },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:21<00:00, 987.80it/s] 


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,2751,0.130919,13.091895
1,5,21013,5871,0.279398,27.939847
2,10,21013,7479,0.355923,35.592252
3,25,21013,11259,0.535811,53.581116


In [17]:
seq_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
        },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:10<00:00, 1955.43it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,5938,0.282587,28.258697
1,5,21013,10549,0.502023,50.202256
2,10,21013,12317,0.586161,58.616095
3,25,21013,16555,0.787846,78.784562


In [ ]:
out = pipeline.infer(three_rscan_q[9000])

In [ ]:
out

PlaceRecognitionResult(descriptor=array([ 0.00143673,  0.00783881,  0.02143787, ...,  0.01978837,
       -0.00375643, -0.0045516 ], shape=(8448,), dtype=float32), indices=array([9000, 8787, 8786, 9001, 9005]), distances=array([8.8449399e-09, 5.0083816e-01, 7.0701253e-01, 7.5864244e-01,
       8.1422371e-01], dtype=float32), db_idx=array([9000, 8787, 8786, 9001, 9005]), db_pose=array([[ 0.798643  ,  0.983432  , -0.132033  ,  0.75250036, -0.5780175 ,
        -0.25381622, -0.18766014],
       [ 0.356377  ,  1.48875   , -0.131014  ,  0.8410546 , -0.427336  ,
        -0.3099954 , -0.11795727],
       [ 0.382755  ,  1.40634   , -0.0807452 ,  0.8403163 , -0.41481936,
        -0.32412416, -0.12937145],
       [ 0.841705  ,  0.995071  , -0.140846  ,  0.7504945 , -0.58146584,
        -0.24821363, -0.19247201],
       [ 0.860654  ,  0.989988  , -0.125784  ,  0.7427527 , -0.56332934,
        -0.28782725, -0.21939473]], dtype=float32))